In [86]:
import torch
from torch import nn


In [87]:
class SmallNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.W1 = nn.Parameter(torch.randn(2, 1))
        self.b1 = nn.Parameter(torch.zeros(2))

        self.W2 = nn.Parameter(torch.randn(2, 2))
        self.b2 = nn.Parameter(torch.zeros(2))

        self.W3 = nn.Parameter(torch.randn(1, 2))
        self.b3 = nn.Parameter(torch.zeros(1))

    def forward(self, x):

        z1 = x @ self.W1.T + self.b1
        a1 = torch.tanh(z1)

        z2 = a1 @ self.W2.T + self.b2
        a2 = torch.tanh(z2)

        y_hat = a2 @ self.W3.T + self.b3

        return y_hat


In [88]:
# Create _ONE_ training example
x = torch.tensor([[0.5]])
y = torch.tensor([[0.8]])
x.shape, y.shape

(torch.Size([1, 1]), torch.Size([1, 1]))

In [89]:
# Create the model and inspect its parameters
model = SmallNet()
print(model)
SmallNet()

for name, parameter in model.named_parameters():
    print(name, parameter, parameter.shape)


SmallNet()
W1 Parameter containing:
tensor([[-0.1617],
        [-0.1976]], requires_grad=True) torch.Size([2, 1])
b1 Parameter containing:
tensor([0., 0.], requires_grad=True) torch.Size([2])
W2 Parameter containing:
tensor([[ 0.5644, -0.9175],
        [ 0.1565,  1.3426]], requires_grad=True) torch.Size([2, 2])
b2 Parameter containing:
tensor([0., 0.], requires_grad=True) torch.Size([2])
W3 Parameter containing:
tensor([[-1.0554,  0.6741]], requires_grad=True) torch.Size([1, 2])
b3 Parameter containing:
tensor([0.], requires_grad=True) torch.Size([1])


In [90]:
# Do ONE forward pass
y_hat = model(x)

print("prediction:", y_hat)


prediction: tensor([[-0.1442]], grad_fn=<AddBackward0>)


In [91]:
# Calculate the loss
loss_fn = nn.MSELoss()
loss = loss_fn(y_hat, y)
print("prediction:", y_hat.item())
print("target:", y.item())
print("loss:", loss.item())
print("loss type:", type(loss))
print("loss size:", loss.size())
print("loss shape:", loss.shape)


prediction: -0.14420855045318604
target: 0.800000011920929
loss: 0.8915297985076904
loss type: <class 'torch.Tensor'>
loss size: torch.Size([])
loss shape: torch.Size([])


In [92]:
# Manually calculate the loss, this should match the value of "loss" from the cell above
error_square = (y.item() - y_hat.item())**2
print(f"MSE: {error_square}")

MSE: 0.891529809260593


In [93]:
# Inspect .grad before backward()
model.W1.grad, model.W2.grad, model.W3.grad

model.b1.grad, model.b2.grad, model.b3.grad


(None, None, None)

In [94]:
# Call loss.backward()
loss.backward()

for name, parameter in model.named_parameters():
    print(name)
    print("value:")
    print(parameter)
    print("gradient:")
    print(parameter.grad)
    print()


W1
value:
Parameter containing:
tensor([[-0.1617],
        [-0.1976]], requires_grad=True)
gradient:
tensor([[ 0.4608],
        [-1.7324]])

b1
value:
Parameter containing:
tensor([0., 0.], requires_grad=True)
gradient:
tensor([ 0.9216, -3.4649])

W2
value:
Parameter containing:
tensor([[ 0.5644, -0.9175],
        [ 0.1565,  1.3426]], requires_grad=True)
gradient:
tensor([[-0.1604, -0.1958],
        [ 0.1005,  0.1227]])

b2
value:
Parameter containing:
tensor([0., 0.], requires_grad=True)
gradient:
tensor([ 1.9890, -1.2466])

W3
value:
Parameter containing:
tensor([[-1.0554,  0.6741]], requires_grad=True)
gradient:
tensor([[-0.0846,  0.2716]])

b3
value:
Parameter containing:
tensor([0.], requires_grad=True)
gradient:
tensor([-1.8884])



In [95]:
# Look at the gradient shapes
for name, parameter in model.named_parameters():
    print(
        name,
        "parameter shape =", parameter.shape,
        "gradient shape  =", parameter.grad.shape
    )


W1 parameter shape = torch.Size([2, 1]) gradient shape  = torch.Size([2, 1])
b1 parameter shape = torch.Size([2]) gradient shape  = torch.Size([2])
W2 parameter shape = torch.Size([2, 2]) gradient shape  = torch.Size([2, 2])
b2 parameter shape = torch.Size([2]) gradient shape  = torch.Size([2])
W3 parameter shape = torch.Size([1, 2]) gradient shape  = torch.Size([1, 2])
b3 parameter shape = torch.Size([1]) gradient shape  = torch.Size([1])


In [96]:
# Connect this to optimizer.step()

print("Old weights before optimizer.step() is called:\n")

for name, parameter in model.named_parameters():
    #print(name, parameter, parameter.grad)
    print(name)
    print("value:")
    print(parameter)
    print("gradient:")
    print(parameter.grad)
    print()


print("\n\nNew weights manually calculate:\n")

lr = 0.1
for name, parameter in model.named_parameters():
    print(name, parameter - lr *parameter.grad)

optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
optimizer.step()

print("\n\nNew weights updated by optimizer.step():\n")
for name, parameter in model.named_parameters():
    print(name, parameter)


Old weights before optimizer.step() is called:

W1
value:
Parameter containing:
tensor([[-0.1617],
        [-0.1976]], requires_grad=True)
gradient:
tensor([[ 0.4608],
        [-1.7324]])

b1
value:
Parameter containing:
tensor([0., 0.], requires_grad=True)
gradient:
tensor([ 0.9216, -3.4649])

W2
value:
Parameter containing:
tensor([[ 0.5644, -0.9175],
        [ 0.1565,  1.3426]], requires_grad=True)
gradient:
tensor([[-0.1604, -0.1958],
        [ 0.1005,  0.1227]])

b2
value:
Parameter containing:
tensor([0., 0.], requires_grad=True)
gradient:
tensor([ 1.9890, -1.2466])

W3
value:
Parameter containing:
tensor([[-1.0554,  0.6741]], requires_grad=True)
gradient:
tensor([[-0.0846,  0.2716]])

b3
value:
Parameter containing:
tensor([0.], requires_grad=True)
gradient:
tensor([-1.8884])



New weights manually calculate:

W1 tensor([[-0.2077],
        [-0.0243]], grad_fn=<SubBackward0>)
b1 tensor([-0.0922,  0.3465], grad_fn=<SubBackward0>)
W2 tensor([[ 0.5805, -0.8980],
        [ 0.1464,  